# Clustering Data Polutan (68 Fitur vs PCA 37 Fitur)
Notebook ini bertujuan untuk melakukan K-Means Clustering dan membandingkan hasilnya antara menggunakan data asli (68 fitur) dan data yang sudah direduksi menjadi 37 fitur.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Memuat Data
df = pd.read_csv('ekstraksi_fitur_co.csv')

print("Dimensi data awal:", df.shape)
df.head()

In [ ]:
# Pemrosesan Data
# Memilih hanya kolom angka untuk dilakukan clustering
X = df.select_dtypes(include=[np.number])

# Menyamakan skala data agar seimbang
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dimensi data setelah disamakan skalanya:", X_scaled.shape)

## 1. Menentukan Jumlah Cluster K-Means pada Data Asli (68 Fitur)
Karena algoritma K-Means membutuhkan input jumlah cluster di awal, kita akan melakukan perulangan untuk mencoba membagi data ke dalam 2 kelompok, 3 kelompok, dan seterusnya hingga 10 kelompok.
Dari percobaan tersebut, kita akan menilai "skor kualitas" pengelompokannya untuk menentukan mana jumlah kelompok yang paling baik.

In [ ]:
skor_kualitas = []
K = range(2, 11) # Mencoba jumlah cluster dari 2 sampai 10

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    
    # Menghitung seberapa baik data ini di-cluster (kualitas kelompok)
    skor = silhouette_score(X_scaled, kmeans.labels_)
    skor_kualitas.append(skor)

# Membuat grafik visualisasi kualitas pengelompokan
plt.figure(figsize=(8, 4))
plt.plot(K, skor_kualitas, 'bo-')
plt.xlabel('Jumlah Cluster')
plt.ylabel('Skor Kualitas Cluster')
plt.title('Evaluasi Jumlah Cluster Terbaik (Data Asli 68 Fitur)')
plt.grid(True)
plt.show()

# Menampilkan hasil skor
for k, skor in zip(K, skor_kualitas):
    print(f"Jumlah Cluster: {k} -> Skor Kualitas: {skor:.4f}")

## 2. Reduksi Dimensi dengan PCA (Menjadi 37 Fitur)
Selanjutnya kita akan memperkecil jumlah fitur dari 68 menjadi 37 fitur agar data lebih padat dan lebih ringan untuk di-cluster.

In [ ]:
# Proses memperkecil fitur (reduksi dimensi) menjadi 37
pca = PCA(n_components=37, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Dimensi data setelah direduksi menjadi:", X_pca.shape)

## 3. Menentukan Jumlah Cluster K-Means pada Data Hasil PCA (37 Fitur)
Kita lakukan langkah pencarian yang sama persis seperti sebelumnya pada data yang fiturnya sudah dikurangi menjadi 37.

In [ ]:
skor_kualitas_pca = []

for k in K:
    kmeans_pca = KMeans(n_clusters=k, random_state=42)
    kmeans_pca.fit(X_pca)
    
    # Menghitung seberapa baik data ini di-cluster pada hasil PCA
    skor = silhouette_score(X_pca, kmeans_pca.labels_)
    skor_kualitas_pca.append(skor)

# Membuat grafik visualisasi kualitas pengelompokan
plt.figure(figsize=(8, 4))
plt.plot(K, skor_kualitas_pca, 'ro-')
plt.xlabel('Jumlah Cluster')
plt.ylabel('Skor Kualitas Cluster')
plt.title('Evaluasi Jumlah Cluster Terbaik (Data Hasil PCA 37 Fitur)')
plt.grid(True)
plt.show()

# Menampilkan hasil skor
for k, skor in zip(K, skor_kualitas_pca):
    print(f"Jumlah Cluster: {k} -> Skor Kualitas: {skor:.4f}")

## Kesimpulan Jawaban

1. **Menentukan jumlah cluster terbaik pada K-Means**:
   - Lihatlah hasil *Skor Kualitas Cluster* (baik dari data asli maupun data 37 fitur). Jumlah cluster (kelompok) yang menghasilkan **skor tertinggi** adalah jumlah cluster yang paling optimal/terbaik untuk dipilih.

2. **Berapa baiknya data ini di-cluster**:
   - Kualitas atau tingkat kebaikan hasil cluster ditunjukkan oleh *Skor Kualitas* itu sendiri.
   - Semakin besar nilainya (maksimal 1.0), berarti data berhasil dipisahkan dan dikelompokkan dengan sangat baik.
   - Semakin kecil nilainya (mendekati 0), berarti hasil pengelompokan kurang bagus karena banyak data yang mirip/bertumpuk sehingga sulit dipisahkan.

3. **Perbandingan (68 Fitur Asli vs 37 Fitur PCA)**:
   - Untuk mengetahui mana yang lebih baik antara data asli (68) dan data reduksi (37), Anda cukup membandingkan skor tertingginya.
   - Jika nilai *Skor Kualitas* pada pengujian PCA (37 fitur) lebih besar daripada data asli, berarti proses pengurangan dimensi (PCA) telah berhasil membuang informasi yang tidak penting dan membuat pengelompokan K-Means menjadi lebih baik.

## Analisis K-Means Menggunakan KNIME

Selain menggunakan Python, proses clustering ini juga divisualisasikan dan diuji menggunakan **KNIME Analytics Platform** dengan alur (*workflow*) membagi dua jalur (Jalur Asli dan Jalur PCA).

**1. Struktur Workflow KNIME**

![knime](../img/cluster1.png)
![knime](../img/cluster2.png)

- **Jalur Atas (Tanpa PCA):** Data mentah dibaca lalu langsung diproses menggunakan node `k-Means` untuk dikelompokkan berdasarkan keseluruhan 68 fitur. Kualitas pengelompokan ini kemudian diukur menggunakan node `Silhouette Coefficient` dan dapat dilihat hasil baris per barisnya pada `Table View`.

- **Jalur Bawah (Dengan PCA):** Data mentah terlebih dahulu masuk ke node `PCA` untuk direduksi dimensinya menjadi 37 fitur utama. Setelah fitur mengecil, data baru dimasukkan ke node `k-Means` lalu dievaluasi skor Silhouette-nya pada `Table View` yang kedua.

**2. Pembahasan Hasil Clustering (Berdasarkan Tabel Visualisasi KNIME)**

Berdasarkan tangkapan layar `Table View` yang dieksekusi, kita dapat menarik kesimpulan berikut:

![knime](../img/cluster3.png)
- **Tabel Pertama (Jalur Atas - Data Asli 68 Fitur):**
  Terlihat bahwa hampir seluruh baris data (seperti daerah Wonokromo, Dukun, Kwanyar, dsb.) mendapatkan **Silhouette Coefficient bernilai 1.0 (sempurna)**. Nilai 1.0 berarti titik data daerah tersebut sangat padat dan berada tepat di tengah-tengah clusternya sendiri, terpisah jauh dari cluster lain. Terdapat pengecualian kecil pada beberapa data (misal Kamal, Bangkalan di `cluster_0`) yang mendapat skor **0.0**, artinya titik daerah tersebut berada tepat di ambang batas antara dua cluster yang saling bersinggungan.

![knime](../img/cluster4.png)
- **Tabel Kedua (Jalur Bawah - Data Hasil PCA 37 Fitur):**
  Pada jalur hasil reduksi PCA, kualitas clustering terbukti sangat stabil dan mempertahankan struktur dari data asli. Mayoritas daerah tetap mencetak skor **1.0**. Ada sedikit penyesuaian skor pada daerah tertentu (seperti Cerme Gresik dan Kec. Kalianget Sumenep yang skornya menjadi **0.998**). 

**Kesimpulan Keseluruhan:**
Hasil visualisasi KNIME membuktikan bahwa membuang 31 fitur dan menyisakan **37 fitur PCA** tidak merusak kualitas data. Kelompok (cluster) yang dihasilkan tetap sangat kuat dan akurat (skor mayoritas mendekati 1.0). Ini menunjukkan PCA sangat efektif dalam meringankan beban program komputer sembari mempertahankan inti informasi dari data polutan tersebut.